# CodeTune v2 — SFT Training

**Environment**: Google Colab Pro A100 80GB  
**Model**: Qwen3.5-9B (bf16 LoRA, LiveCodeBench 65.6)

Run cells in order. Exp A → B → C.

In [ ]:
import subprocess, torch
result = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True)
print("GPU:", result.stdout.strip())
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB, BF16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# unsloth MUST be first
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl peft accelerate bitsandbytes -q
!pip install datasets huggingface-hub python-dotenv pyyaml click -q
print("Done.")

In [ ]:
from huggingface_hub import login
import wandb

HF_TOKEN      = "YOUR_HF_TOKEN"
HF_USERNAME   = "Michlitt"
WANDB_API_KEY = ""   # ← 把你的 W&B API key 粘贴在这里

login(token=HF_TOKEN)
print("Logged in as", HF_USERNAME)

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print("W&B logged in")
else:
    import os
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled (no key set)")

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
REPO_ID = f"{HF_USERNAME}/codetune-v2-sft"
for p in ["data/processed/targeted","configs","results/sft_checkpoints","scripts"]:
    Path(p).mkdir(parents=True, exist_ok=True)
for fname in ["sft_train.jsonl","sft_val.jsonl"]:
    loc = hf_hub_download(repo_id=REPO_ID,filename=fname,repo_type="dataset",local_dir="data/processed")
    print(f"{fname}: {sum(1 for l in open(loc) if l.strip()):,}")
loc = hf_hub_download(repo_id=REPO_ID,filename="targeted/targeted_filtered.jsonl",repo_type="dataset",local_dir="data/processed")
print(f"targeted: {sum(1 for l in open(loc) if l.strip())}")

In [ ]:
from pathlib import Path
Path("scripts").mkdir(exist_ok=True)
Path("configs").mkdir(exist_ok=True)
Path("scripts/sft_train.py").write_text('''\
"""SFT training script — run on Colab A100 80GB."""

from __future__ import annotations
import json, os, sys
from pathlib import Path
import click, yaml
from dotenv import load_dotenv

load_dotenv()

ROOT = Path(__file__).parent.parent
SFT_CFG_PATH      = ROOT / "configs/sft_config.yaml"
ABLATION_CFG_PATH = ROOT / "configs/ablation_matrix.yaml"

SYSTEM_PROMPT = (
    "You are an expert Python programmer. Write clean, efficient, and correct code. "
    "Always handle edge cases and include brief comments for complex logic."
)


def _load_yaml(path):
    return yaml.safe_load(path.read_text(encoding="utf-8")) if path.exists() else {}


def _load_jsonl(path):
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}\\nRun scripts/prepare_sft_data.py first.")
    results = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return results


def _filter_by_sources(samples, sources):
    if not sources:
        return samples
    return [s for s in samples if s.get("source", "") in sources]


@click.command()
@click.option("--exp-id", default="sft_C_with_targeted", show_default=True)
@click.option("--data-sources", default=None)
@click.option("--lora-r", default=None, type=int)
@click.option("--epochs", default=None, type=int)
@click.option("--resume-from", default=None)
def main(exp_id, data_sources, lora_r, epochs, resume_from):
    try:
        import torch
        from unsloth import FastLanguageModel
        from trl import SFTTrainer, SFTConfig
        from datasets import Dataset
    except ImportError as e:
        print(f"ERROR: Missing dependency — {e}")
        sys.exit(1)

    base_cfg  = _load_yaml(SFT_CFG_PATH)
    ablation  = _load_yaml(ABLATION_CFG_PATH).get("experiments", {})
    exp_cfg   = ablation.get(exp_id, {})

    sources  = data_sources.split(",") if data_sources else exp_cfg.get("data_sources", ["magicoder", "evol", "targeted"])
    r        = lora_r or exp_cfg.get("lora_r", base_cfg.get("lora", {}).get("r", 64))
    n_epochs = epochs or exp_cfg.get("num_train_epochs", base_cfg.get("training", {}).get("num_train_epochs", 1))

    model_name  = base_cfg["model"]["name"]
    max_seq_len = base_cfg["model"]["max_seq_length"]
    train_cfg   = base_cfg["training"]
    ckpt_cfg    = base_cfg["checkpointing"]
    lora_cfg    = base_cfg["lora"]

    out_dir = ROOT / ckpt_cfg["output_dir"] / exp_id
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\\n{\'=\'*60}")
    print(f"SFT Training — {exp_id}")
    print(f"{\'=\'*60}")
    print(f"Model      : {model_name}")
    print(f"LoRA rank  : {r}")
    print(f"Epochs     : {n_epochs}")
    print(f"Sources    : {sources}")
    print(f"Output dir : {out_dir}\\n")

    train_path = ROOT / base_cfg["data"]["train_file"]
    val_path   = ROOT / base_cfg["data"]["val_file"]
    train_raw  = _filter_by_sources(_load_jsonl(train_path), sources)
    val_raw    = _filter_by_sources(_load_jsonl(val_path),   sources)
    print(f"Train samples: {len(train_raw):,}  Val samples: {len(val_raw):,}")

    print(f"\\nLoading {model_name} …")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_len,
        load_in_4bit=base_cfg["model"]["load_in_4bit"],
        dtype=getattr(torch, base_cfg["model"]["dtype"]),
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=r,
        target_modules=lora_cfg["target_modules"],
        lora_alpha=r * 2,
        lora_dropout=lora_cfg["lora_dropout"],
        bias=lora_cfg["bias"],
        use_gradient_checkpointing=lora_cfg["use_gradient_checkpointing"],
        random_state=lora_cfg["random_state"],
    )
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")

    def format_fn(example):
        return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)}

    train_ds = Dataset.from_list(train_raw).map(format_fn, remove_columns=["messages", "source", "task_id"])
    val_ds   = Dataset.from_list(val_raw).map(format_fn,   remove_columns=["messages", "source", "task_id"])

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=SFTConfig(
            output_dir=str(out_dir),
            per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
            gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
            num_train_epochs=n_epochs,
            learning_rate=train_cfg["learning_rate"],
            lr_scheduler_type=train_cfg["lr_scheduler_type"],
            warmup_ratio=train_cfg["warmup_ratio"],
            weight_decay=train_cfg["weight_decay"],
            max_grad_norm=train_cfg["max_grad_norm"],
            bf16=train_cfg["bf16"],
            tf32=train_cfg.get("tf32", True),
            optim=train_cfg["optim"],
            dataloader_num_workers=train_cfg.get("dataloader_num_workers", 0),
            seed=train_cfg["seed"],
            eval_strategy="steps",
            logging_steps=ckpt_cfg["logging_steps"],
            eval_steps=ckpt_cfg["eval_steps"],
            save_steps=ckpt_cfg["save_steps"],
            save_total_limit=ckpt_cfg["save_total_limit"],
            load_best_model_at_end=ckpt_cfg["load_best_model_at_end"],
            metric_for_best_model=ckpt_cfg["metric_for_best_model"],
            report_to=ckpt_cfg.get("report_to", "none"),
            dataset_text_field="text",
            max_seq_length=max_seq_len,
            run_name=exp_id,
        ),
    )

    print("\\nRunning pre-training eval …")
    trainer.evaluate()

    print(f"\\nStarting training ({n_epochs} epoch(s)) …")
    trainer.train(resume_from_checkpoint=resume_from)

    final_dir = out_dir / "final"
    final_dir.mkdir(exist_ok=True)
    model.save_pretrained(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))

    import json as _json
    meta = {"exp_id": exp_id, "model_name": model_name, "lora_r": r, "num_epochs": n_epochs,
            "data_sources": sources, "train_samples": len(train_raw), "val_samples": len(val_raw)}
    (final_dir / "experiment_meta.json").write_text(_json.dumps(meta, indent=2), encoding="utf-8")

    print(f"\\nModel saved to {final_dir}")
    print(f"Training complete for experiment: {exp_id}")


if __name__ == "__main__":
    main()
''', encoding="utf-8")
Path("configs/sft_config.yaml").write_text("""\
model:
  name: "unsloth/Qwen3.5-9B"
  max_seq_length: 2048
  load_in_4bit: false
  dtype: "bfloat16"

lora:
  r: 64
  lora_alpha: 128
  lora_dropout: 0.05
  bias: "none"
  target_modules: [q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj]
  use_gradient_checkpointing: "unsloth"
  random_state: 42

training:
  per_device_train_batch_size: 8
  gradient_accumulation_steps: 2
  num_train_epochs: 1
  learning_rate: 2.0e-4
  lr_scheduler_type: "cosine"
  warmup_ratio: 0.05
  weight_decay: 0.01
  max_grad_norm: 1.0
  bf16: true
  tf32: true
  optim: "adamw_8bit"
  dataloader_num_workers: 4
  seed: 42

checkpointing:
  output_dir: "results/sft_checkpoints"
  logging_steps: 10
  eval_steps: 100
  save_steps: 200
  save_total_limit: 3
  load_best_model_at_end: true
  metric_for_best_model: "eval_loss"
  report_to: "wandb"

data:
  train_file: "data/processed/sft_train.jsonl"
  val_file:   "data/processed/sft_val.jsonl"
  dataset_text_field: "text"
""", encoding="utf-8")
Path("configs/ablation_matrix.yaml").write_text("""\
experiments:
  sft_A_magicoder_only:
    description: "Baseline: Magicoder-OSS-Instruct only"
    data_sources: [magicoder]
    lora_r: 64
    num_train_epochs: 1
  sft_B_full_data:
    description: "Extended: Magicoder + EvolCodeAlpaca"
    data_sources: [magicoder, evol]
    lora_r: 64
    num_train_epochs: 1
  sft_C_with_targeted:
    description: "Full + Targeted (primary)"
    data_sources: [magicoder, evol, targeted]
    lora_r: 64
    num_train_epochs: 1
  sft_D_rank32:
    description: "Rank ablation: r=32 vs r=64"
    data_sources: [magicoder, evol, targeted]
    lora_r: 32
    num_train_epochs: 1
  sft_E_epoch1:
    description: "Epoch ablation: 1 epoch"
    data_sources: [magicoder, evol, targeted]
    lora_r: 64
    num_train_epochs: 1
""", encoding="utf-8")
print("Scripts ready.")

In [ ]:
# Exp A: magicoder only (baseline)
import sys, runpy
sys.argv = ["sft_train.py", "--exp-id", "sft_A_magicoder_only", "--data-sources", "magicoder"]
try:
    runpy.run_path("scripts/sft_train.py", run_name="__main__")
except SystemExit as e:
    print(f"Done (exit {e.code})")

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
repo = f"{HF_USERNAME}/codetune-v2-sft-A"
try: api.create_repo(repo_id=repo,repo_type="model",private=True)
except: pass
api.upload_folder(folder_path="results/sft_checkpoints/sft_A_magicoder_only/final",repo_id=repo,repo_type="model")
print("Uploaded:", repo)

In [ ]:
# Exp B: magicoder + evol
import sys, runpy
sys.argv = ["sft_train.py", "--exp-id", "sft_B_full_data", "--data-sources", "magicoder,evol"]
try:
    runpy.run_path("scripts/sft_train.py", run_name="__main__")
except SystemExit as e:
    print(f"Done (exit {e.code})")

In [ ]:
# Exp C: full + targeted (PRIMARY)
import sys, runpy
sys.argv = ["sft_train.py", "--exp-id", "sft_C_with_targeted", "--data-sources", "magicoder,evol,targeted"]
try:
    runpy.run_path("scripts/sft_train.py", run_name="__main__")
except SystemExit as e:
    print(f"Done (exit {e.code})")

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
for exp_id, suffix in [("sft_B_full_data","B"),("sft_C_with_targeted","C")]:
    repo = f"{HF_USERNAME}/codetune-v2-sft-{suffix}"
    try: api.create_repo(repo_id=repo,repo_type="model",private=True)
    except: pass
    api.upload_folder(folder_path=f"results/sft_checkpoints/{exp_id}/final",repo_id=repo,repo_type="model")
    print("Uploaded:", repo)